> **2026** — local-first biology (Biopython/PDB/pandas); optional paid LLM only where noted. See `UPDATE_2026.md`.

# Chapter 7 — Protein Structure Evidence (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2007.%20LangChain%20for%20Biology/LC4LSH_Chapter_7_Protein_Structure_Evidence.ipynb)

**Learning objectives**
- Summarize a PDB/mmCIF record (chains, ligands, resolution)
- Distinguish experimental vs predicted structure status
- Read confidence fields (pLDDT) for predicted models
- Keep a clear record of structure provenance

> Runtime: ~5 min (local; optional network fetch)  
> Cost: free  
> Data: small built-in PDB-style records


## Environment setup


### Secrets (optional LLM only)


In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False
if not IN_COLAB:
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass

def get_secret(name, default=None):
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)

# These notebooks run locally (Biopython/pandas); a paid LLM is OPTIONAL for narrative only.
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"
if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LS_OPENAI_API_KEY", "sk-...")
print("Optional LLM provider:", API_KEY_PROVIDER, "(analysis runs without it)")
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""


### Install pinned dependencies


In [ ]:
%pip install -q biopython "pandas>=2.0" "numpy>=1.26" "matplotlib>=3.8" "scikit-learn>=1.4" "langchain==1.0.0" "langchain-openai==1.0.0" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)


In [ ]:
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter7-protein-structure"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF (fine — these notebooks are local-first)")


## Experimental vs predicted structures

**Experimental** structures (X-ray/NMR/cryo-EM) carry a measured resolution and experimental metadata. **Predicted** models (AlphaFold etc.) carry per-residue confidence (pLDDT). They must never be treated as interchangeable.


## 1. Minimal PDB header summary


In [ ]:
PDB = """HEADER    TRANSFERASE (KINASE)                    01-JAN-00   9ABC
TITLE     EXAMPLE KINASE DOMAIN
EXPDTA    X-RAY DIFFRACTION
REMARK   2 RESOLUTION.    1.80 ANGSTROMS.
HELIX    1   1 ALA A   10  GLY A   20  1                                  11
ATOM      1  N   ALA A  10      10.0   5.0   2.0  1.00 15.00           N
ATOM      2  CA  ALA A  10      11.0   5.5   2.5  1.00 15.00           C
ATOM      3  N   GLY A  11      12.0   6.0   3.0  1.00 14.00           N
ATOM      4  N   ALA B  10      10.5   5.2   2.2  1.00 16.00           N
HETATM 9000  O   HOH A 500      20.0  10.0   8.0  1.00 20.00           O
HETATM 9001  C1  LIG A 600      25.0  12.0   9.0  1.00 18.00           C
END
"""
def pdb_summary(text):
    info = {"chains": set(), "expdta": None, "resolution": None, "ligands": set(), "het_res": set()}
    for line in text.splitlines():
        rec = line[:6].strip()
        if rec == "EXPDTA":
            info["expdta"] = line[10:].strip()
        elif rec == "REMARK" and "RESOLUTION." in line and "ANGSTROMS" in line:
            try:
                info["resolution"] = float(line.split("RESOLUTION.")[1].split("ANGSTROMS")[0].strip())
            except Exception:
                pass
        elif rec == "ATOM":
            info["chains"].add(line[21])
        elif rec == "HETATM":
            resname = line[17:20].strip()
            info["het_res"].add(resname)
            if resname not in ("HOH",):
                info["ligands"].add(resname)
    info["chains"] = sorted(info["chains"])
    info["ligands"] = sorted(info["ligands"])
    return info

s = pdb_summary(PDB)
print("method:", s["expdta"], "| resolution:", s["resolution"])
print("chains:", s["chains"], "| non-water ligands:", s["ligands"], "| het residues:", sorted(s["het_res"]))


## 2. Experimental status flag


In [ ]:
def structure_status(expdta):
    if not expdta:
        return "unknown"
    e = expdta.upper()
    if "THEORETICAL" in e or "PREDICTED" in e or "MODEL" in e:
        return "predicted"
    if any(k in e for k in ("X-RAY", "NMR", "ELECTRON", "NEUTRON", "FIBER")):
        return "experimental"
    return "unknown"

print(s["expdta"], "->", structure_status(s["expdta"]))
for m in ["X-RAY DIFFRACTION", "SOLUTION NMR", "ELECTRON MICROSCOPY", "THEORETICAL MODEL"]:
    print(m, "->", structure_status(m))


## 3. Predicted-model confidence (pLDDT in B-factor)


In [ ]:
# In AlphaFold-style PDB/mmCIF, per-residue pLDDT is stored in the B-factor column.
def mean_plddt(text):
    vals = []
    for line in text.splitlines():
        if line.startswith("ATOM"):
            try:
                vals.append(float(line[60:66]))
            except Exception:
                pass
    return round(sum(vals) / len(vals), 1) if vals else None

def plddt_band(v):
    if v is None: return "n/a"
    if v >= 90: return "very high"
    if v >= 70: return "confident"
    if v >= 50: return "low"
    return "very low"

mp = mean_plddt(PDB)
print("mean pLDDT:", mp, "->", plddt_band(mp))


## 4. Optional: fetch a real PDB entry (network)


In [ ]:
import urllib.request

def fetch_pdb_header(pdb_id="1CRN"):
    url = f"https://files.rcsb.org/header/{pdb_id.upper()}.pdb"
    try:
        txt = urllib.request.urlopen(url, timeout=10).read().decode()
        return pdb_summary(txt)
    except Exception as e:
        return {"error": str(e)}

# res = fetch_pdb_header("1CRN")  # uncomment when online
# print(res)
print("Network fetch is optional; the parser works offline on local files.")


## 5. Provenance record


In [ ]:
provenance = {
    "source": "local PDB text",
    "experimental_status": structure_status(s["expdta"]),
    "resolution_A": s["resolution"],
    "mean_plddt": mp,
    "chains": s["chains"],
    "ligands": s["ligands"],
}
import json
print(json.dumps(provenance, indent=2))


## Limitations & safety notes

- This parser is illustrative (fixed-column PDB); use `Bio.PDB`/gemmi for production.
- pLDDT measures **local confidence**, not global correctness or biological relevance.
- A high-confidence or high-resolution structure is not evidence of function/mechanism by itself.
- Local/free; optional network fetch for real entries.


In [ ]:
# Cleanup
import gc
for _v in ("records", "df", "model", "llm", "structure", "X"):
    globals().pop(_v, None)
gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Why separate experimental from predicted structures?</summary>They have different error models and confidence semantics; mixing them leads to over-trusting predictions.</details>

<details><summary>What does pLDDT NOT tell you?</summary>It does not tell you whether the fold is functionally relevant or whether the whole model is globally correct.</details>

<details><summary>Why track resolution?</summary>Resolution bounds how precisely atom positions are known; low resolution limits fine-grained conclusions.</details>

### Tasks
- **Task A** - Use `Bio.PDB` to compute chain lengths and a residue-type histogram for a real PDB file.
- **Task B** - Parse an mmCIF file (gemmi or Bio.PDB.MMCIFParser) and extract the same provenance record.
- **Task C** - Flag low-pLDDT regions (<70) and list their residue ranges.
- **Task D** - Build a small structure-QC report combining status, resolution/pLDDT, and ligand inventory.
